# Full preprocessing pipeline for the supervised transformer

End-to-end: raw ACLED export → cleaned text → trigger-word labels → balanced
20-class training file consumed by `transformers_model_finetuner.py`.

- **Sections 1–2** reproduce the only parts of `data_cleaning.ipynb` and
  `labeling_triggerwords.ipynb` needed to produce the model's input.
- **Sections 3–6** drop non-categories and balance the labels into
  `labeled_balanced_20.csv`.

Two deliberate choices in the balancing stage, both required for the model to reach
the target taxonomy:

1. **No category merging.** Unlike `dataset_balancing.ipynb` (which merges the fine
   classes into a coarse 15-class set via `class_mapping`), we keep every original
   class so the model can predict the full taxonomy.
2. **Drop `NoN` and `unknown`.** `unknown` is not a real category — the model exists
   precisely to assign the unknown events to real classes — so training on it would
   teach the model to emit `unknown`. Dropping it yields exactly 20 classes.

Paths are relative to `pipeline/`.

In [ ]:
import re
import nltk
import pandas as pd
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
import os

DATA_DIR = '../data'
os.makedirs(DATA_DIR, exist_ok=True)


RAW_CSV = f'{DATA_DIR}/ACLED Data_2026-06-10.csv'
EVENTS_CSV = f'{DATA_DIR}/filtered_events.csv'
EVENTS_CC_CSV = f'{DATA_DIR}/filtered_events_country_code.csv'
LABELED_CSV = f'{DATA_DIR}/labeled.csv'
OUTPUT_CSV = f'{DATA_DIR}/labeled_balanced_20.csv'

# British -> American spelling normalization. Applied uniformly (in preprocess() and in
# classify()) before any matching, the same way .lower() standardizes case - so triggers
# can be written in one spelling ('labor', 'criminalize') and still match British notes
# ('labour', 'criminalise'). We map specific, unambiguous stems via substring replace
# rather than blanket suffix rules (-our/-ise), which would corrupt words like 'tourism'
# or 'rise'. Substrings cover inflections too: 'labour' -> labour/labours/labourers.
BRITISH_TO_US = {
    'labour': 'labor', 'criminalis': 'criminaliz', 'organis': 'organiz',
    'mobilis': 'mobiliz', 'nationalis': 'nationaliz', 'privatis': 'privatiz',
    'recognis': 'recogniz', 'neighbour': 'neighbor', 'behaviour': 'behavior',
    'colour': 'color', 'honour': 'honor', 'favour': 'favor',
    'defence': 'defense', 'licence': 'license', 'centre': 'center', 'theatre': 'theater',
}

def normalize_spelling(text):
    """Lowercased text in, British spellings rewritten to American."""
    for brit, us in BRITISH_TO_US.items():
        text = text.replace(brit, us)
    return text

## 1. Clean the raw ACLED export

From `data_cleaning.ipynb`. Read the raw export, drop duplicate notes, build the
`clean_notes` column (lowercase, strip the leading date boilerplate, lemmatize, drop
stopwords), keep only European protest/riot events, and write
`filtered_events_country_code.csv`.

In [2]:
df = pd.read_csv(RAW_CSV, delimiter=',', low_memory=False)
print(f'Loaded {len(df)} raw rows')

Loaded 616973 raw rows


In [3]:
df = df.drop_duplicates(subset=['notes', 'event_date', 'country'])
print(f'{len(df)} rows after dropping duplicate notes')

616270 rows after dropping duplicate notes


In [ ]:
nltk.download('wordnet')
nltk.download('stopwords')
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess(text):
    text = normalize_spelling(str(text).lower())
    text = re.sub(r'\W+', ' ', text)
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]
    return ' '.join(tokens)

# Strip the leading "On <date>," boilerplate, then clean and drop the first 3 tokens.
regex_pattern = r"^(?:\[)?.?\d{4}(?:[,.:]?\s)?"
df['notes'] = df['notes'].str.replace(regex_pattern, '', regex=True).str.strip()
df['clean_notes'] = df['notes'].apply(preprocess)
df['clean_notes'] = df['clean_notes'].apply(lambda x: ' '.join(str(x).split()[3:]))
df = df[df['clean_notes'].str.strip() != '']
print(f'{len(df)} rows with non-empty clean_notes')

In [5]:
ndf = df[df['event_type'].isin(['Protests', 'Riots'])]
ndf = ndf[ndf['sub_event_type'] != 'Mob violence']
print(f'{len(ndf)} protest/riot rows')

237651 protest/riot rows


In [6]:
ndf['country_code'] = ndf['event_id_cnty'].str[:3]

european_codes = [
    'ALB', 'AND', 'AUT', 'BEL', 'BGR', 'BIH', 'CHE', 'CYP', 'CZE', 'DEU', 'DNK',
    'ESP', 'EST', 'FIN', 'FRA', 'GBR', 'GRC', 'HRV', 'HUN', 'IRL', 'ISL', 'ITA',
    'KOS', 'LIE', 'LTU', 'LUX', 'LVA', 'MCO', 'MDA', 'MKD', 'MLT', 'MNE', 'NLD',
    'NOR', 'POL', 'PRT', 'ROU', 'SMR', 'SRB', 'SVK', 'SVN', 'SWE', 'VAT'
]
european_countries = [
    'Albania', 'Andorra', 'Austria', 'Belgium', 'Bosnia and Herzegovina', 'Bulgaria',
    'Croatia', 'Cyprus', 'Czech Republic', 'Denmark', 'Estonia', 'Finland', 'France',
    'Germany', 'Greece', 'Hungary', 'Iceland', 'Ireland', 'Italy', 'Kosovo',
    'Latvia', 'Liechtenstein', 'Lithuania', 'Luxembourg', 'Malta', 'Moldova',
    'Monaco', 'Montenegro', 'Netherlands', 'North Macedonia', 'Norway', 'Poland',
    'Portugal', 'Romania', 'San Marino', 'Serbia', 'Slovakia', 'Slovenia',
    'Spain', 'Sweden', 'Switzerland', 'United Kingdom', 'Vatican City'
]

nnndf = ndf[ndf['country_code'].isin(european_codes)]
nndf = ndf[ndf['country'].isin(european_countries)]

nndf.to_csv(EVENTS_CSV, index=False)
nnndf.to_csv(EVENTS_CC_CSV, index=False)
print(f'Wrote {len(nnndf)} rows to {EVENTS_CC_CSV}')

Wrote 214232 rows to ../data/filtered_events_country_code.csv


Horizontal bar chart of how many protest/riot events each European country contributes to the filtered set, sorted from most to fewest.

In [7]:
import plotly.express as px

country_counts = nnndf['country'].value_counts()
fig = px.bar(
    x=country_counts.values, y=country_counts.index, orientation='h',
    text=country_counts.values,
    labels={'x': 'Events', 'y': 'Country'},
)
fig.update_layout(
    title={'text': 'European protest/riot events per country', 'x': 0.5},
    plot_bgcolor='white', width=900, height=800,
    yaxis={'categoryorder': 'total ascending'},
)
fig.show()

## 2. Assign coarse labels with trigger words

From `labeling_triggerwords.ipynb`. Start every event as `unknown`, then assign the
first class whose trigger word appears in the original `notes`. Writes `labeled.csv`,
which carries both the `class` and the `clean_notes` columns that section 3 consumes.

In [8]:
df = pd.read_csv(EVENTS_CC_CSV, low_memory=False)
df['class'] = 'unknown'
print(f'Labeling {len(df)} events')

Labeling 214232 events


In [ ]:
Classes_dic = {
    "blm": ["black lives matter"],
    "lgbtq": ["lgb", "lesbian", "gay", "homosexual", "transsexual", "queer", "homophobia", "transphobia", "biphobia", "trans rights"],
    "women rights": ["women's rights", "feminism", "feminist", "against women", "women protested", "abortion", "sexual violence", "sexual assault", "sexual harassment", "sexual abuse"],
    "immigration": ["migrants", "immigration", "against migration", "deportation detention"],
    "unjust law enforcement": ["police brutality", "criminalize protests", "criminalize demonstrations", "police misconduct", "police repression"],
    "discrimination": ["discrimination", "racism"],
    "climate": ["climate change", "fossil fuels", "greenwashing", "climate agenda", "global warming"],
    "palestine-israel conflict": ["gaza", "palestine", "israel", "hamas", "palestinian"],
    # 'labor conditions' moved to 'labor rights' (it is a labour grievance, not animal welfare).
    "animal welfare": ["species extinction", "animal welfare", "animal rights", "animal protection", "bullfighting", "animals locked", "wildlife", "cruel"],
    "farmers": ["farmers", "agriculture", "agricultural", "intensive farming"],
    # ' pension' has a leading space (like ' wages' / ' russia') so it no longer matches
    # inside 'suspension'. 'labor conditions' moved here from animal welfare. Triggers use
    # US spelling; notes are normalized to US (normalize_spelling) so British spellings match.
    "labor rights": ["labor agreement", " wages", "rights of workers", "labor rights", "higher salaries", "working conditions", "labor conditions", "commission fees", " pension", "salary equalization", "unfairly dismissed"],
    # 'nurseries' dropped: it means childcare/daycare, not health care.
    "health care": ["healthcare", "hospital ", "hospitals", "emergency clinics", "emergency care"],
    # bare 'environment' replaced with 'environmental' / 'the environment' so it no longer
    # matches 'working environment' / 'business environment' in labour protests.
    "environment": ["environmental", "the environment", "pfas", "nitrogen", "planned felling", "biodiversity", "park project"],
    "public services": ["collapse of a concrete canopy", "canopy collapse", " bus ", "traffic accidents", "railway station", "bike lanes", "road connection", "public service", "pedestrianization", "child-safe intersections", "bike street", "play street", "reasonable mobility", "cycling conditions", "urban development", "free transport"],
    "ukraine-russia war": [" russia", "ukrain"],
    "housing": ["residential complex", "dignified housing", "evict"],
    "culture": ["tourism", "tourists", "cultural sector"],
    "policies & politics": ["social welfare", "social services", "social assistance", "economic justice", "economic sovereignty", "economic independence", "adoption of the euro", "euro adoption", "council's plan", "nightlife noise", "municipality", "regional government", "political criticism", "political opposition", "against the pm", "resignation of the president", "political rights", "political prisoners", "anti-eu", "pro-eu", "democratic", "referendums", "urgent elections", "distinct autonomy"],
    "pandemic": ["pandemic", "covid", "coronavirus"],
    "education": ["education", "teacher", "academic", "education", "professor", "university", "student loan"]
}

In [ ]:
def classify(df, classes_dic):
    for index, row in df.iterrows():
        if row["class"] == "unknown":
            notes_lower = normalize_spelling(row["notes"].lower())
            classified = False
            for class_name, words in classes_dic.items():
                for word in words:
                    if word in notes_lower:
                        df.at[index, "class"] = class_name
                        classified = True
                        break
                if classified:
                    break

classify(df, Classes_dic)
print(df["class"].value_counts())

Bar chart of the trigger-word assignment across all events. The gray `unknown` bar is everything no trigger word matched; the colored bars are the events each class captured.

In [11]:
import plotly.express as px

class_counts = df['class'].value_counts()
colors = ['unknown' if c == 'unknown' else 'labeled' for c in class_counts.index]
fig = px.bar(
    x=class_counts.index, y=class_counts.values, color=colors, text=class_counts.values,
    color_discrete_map={'unknown': 'lightgray', 'labeled': 'steelblue'},
    labels={'x': 'Class', 'y': 'Events', 'color': ''},
)
fig.update_layout(
    title={'text': 'Trigger-word labels (unknown = not matched)', 'x': 0.5},
    plot_bgcolor='white', width=900, height=500,
)
fig.show()

In [12]:
df.to_csv(LABELED_CSV)
print(f'Wrote {len(df)} labeled rows to {LABELED_CSV}')

Wrote 214232 labeled rows to ../data/labeled.csv


## 3. Load the labeled data

We only need the label and the cleaned text. `low_memory=False` avoids the mixed-type
warning on the wide source CSV.

In [13]:
dataset = pd.read_csv(LABELED_CSV, low_memory=False)
dataset = dataset[['class', 'clean_notes']]
print(f'Loaded {len(dataset)} rows from {LABELED_CSV}')

Loaded 214232 rows from ../data/labeled.csv


## 4. Drop non-categories and empty text

`NoN` is a placeholder and `unknown` is not a class we want the model to emit. Rows
without `clean_notes` carry no signal and would break tokenization.

In [14]:
dataset = dataset[~dataset['class'].isin(['NoN', 'unknown'])]
dataset = dataset.dropna(subset=['clean_notes'])

print(f'{len(dataset)} rows across {dataset["class"].nunique()} classes after filtering\n')
dataset['class'].value_counts()

130311 rows across 20 classes after filtering



class
labor rights                 22563
pandemic                     20710
farmers                      12907
palestine-israel conflict    11996
education                     9366
women rights                  7597
ukraine-russia war            6493
climate                       5989
environment                   5630
policies & politics           5201
health care                   4556
lgbtq                         3303
public services               3235
immigration                   2645
discrimination                2531
animal welfare                2024
housing                       1032
blm                            926
culture                        806
unjust law enforcement         801
Name: count, dtype: int64

Raw class sizes after dropping non-categories, with the dashed line marking the `CAP_PER_CLASS = 3000` threshold. Bars above the line get undersampled in Section 5; bars below it stay as-is.

In [15]:
import plotly.express as px

class_counts = dataset['class'].value_counts()
fig = px.bar(
    x=class_counts.index, y=class_counts.values, color=class_counts.index, text=class_counts.values,
    labels={'x': 'Class', 'y': 'Rows'},
)
fig.add_hline(y=3000, line_dash='dash', line_color='red', annotation_text='cap = 3000')
fig.update_layout(
    title={'text': 'Class sizes before balancing', 'x': 0.5},
    plot_bgcolor='white', width=900, height=500, showlegend=False,
)
fig.show()

## 5. Balance the classes

Cap each class to at most `CAP_PER_CLASS` rows. This undersamples the dominant classes
(e.g. `labor rights`, `pandemic`) while leaving the smaller ones at their natural size —
it does not oversample. Adjust `CAP_PER_CLASS` to trade dataset size for balance.

In [16]:
CAP_PER_CLASS = 3000

def keep_first_n(df, n):
    return df.groupby('class').head(n)

dataset = keep_first_n(dataset, CAP_PER_CLASS)

print(f'{len(dataset)} rows across {dataset["class"].nunique()} classes after balancing\n')
dataset['class'].value_counts()

49765 rows across 20 classes after balancing



class
environment                  3000
labor rights                 3000
policies & politics          3000
education                    3000
public services              3000
farmers                      3000
women rights                 3000
ukraine-russia war           3000
health care                  3000
palestine-israel conflict    3000
lgbtq                        3000
climate                      3000
pandemic                     3000
immigration                  2645
discrimination               2531
animal welfare               2024
housing                      1032
blm                           926
culture                       806
unjust law enforcement        801
Name: count, dtype: int64

## 6. (Optional) Inspect the class distribution

In [17]:
import plotly.express as px
from plotly.offline import iplot, init_notebook_mode

init_notebook_mode(True)

counts = dataset['class'].value_counts()
fig = px.bar(x=counts.index, y=counts.values, color=counts.index, text=counts.values)
fig.update_traces(hovertemplate="Category:'%{x}' Counted: %{y}")
fig.update_layout(
    title={'text': 'Balanced Category Counts', 'x': 0.5, 'font': {'size': 30}},
    xaxis={'title': 'Category', 'showgrid': False},
    yaxis={'title': 'Count', 'showgrid': False},
    plot_bgcolor='white', width=900, height=500, showlegend=False,
)
iplot(fig)

## 7. Write the training file

This is the file `transformers_model_finetuner.py` (`file_path`) and
`labeling_by_transformer_model.ipynb` (`TRAIN_CSV`) read.

In [18]:
dataset.to_csv(OUTPUT_CSV, index=False)
print(f'Wrote {len(dataset)} rows, {dataset["class"].nunique()} classes to {OUTPUT_CSV}')
print('Classes:', sorted(dataset['class'].unique()))

Wrote 49765 rows, 20 classes to ../data/labeled_balanced_20.csv
Classes: ['animal welfare', 'blm', 'climate', 'culture', 'discrimination', 'education', 'environment', 'farmers', 'health care', 'housing', 'immigration', 'labor rights', 'lgbtq', 'palestine-israel conflict', 'pandemic', 'policies & politics', 'public services', 'ukraine-russia war', 'unjust law enforcement', 'women rights']
